In [1]:
"""
model4_isolation_forest.py
══════════════════════════════════════════════════════════════════════════════
Model 4 — Isolation Forest  (Unsupervised Novelty Detector)
──────────────────────────────────────────────────────────────────────────────
TRAINING PHILOSOPHY
  • Trained EXCLUSIVELY on LOW-threat rows — these represent the "normal"
    background fingerprint of alert activity.
  • At inference, any row that isolates unusually quickly from that learned
    normal space gets a high anomaly score.
  • No labels required at inference time; the model is purely distribution-based.

WHY ISOLATION FOREST FOR THIS TASK
  • Random binary splits isolate outliers in far fewer cuts than normal points.
  • Scales to 50k+ rows with minimal memory.
  • Does not assume any cluster shape or Gaussian distribution — ideal for
    SOCMINT data that mixes integer counts, ratios, and temporal signals.
  • Catches *novel* attack compositions, not just high-severity ones — a
    coordinated campaign can look moderate in severity yet be highly anomalous
    in its feature fingerprint (e.g., unusually pure propaganda with no informants).

OUTPUT
  • anomaly_score  : raw contamination-aware score (higher = more anomalous)
  • if_flag        : 1 = anomalous (novelty), 0 = normal
  • percentile_score : normalised 0-100 rank for dashboard display
══════════════════════════════════════════════════════════════════════════════
"""

'\nmodel4_isolation_forest.py\n══════════════════════════════════════════════════════════════════════════════\nModel 4 — Isolation Forest  (Unsupervised Novelty Detector)\n──────────────────────────────────────────────────────────────────────────────\nTRAINING PHILOSOPHY\n  • Trained EXCLUSIVELY on LOW-threat rows — these represent the "normal"\n    background fingerprint of alert activity.\n  • At inference, any row that isolates unusually quickly from that learned\n    normal space gets a high anomaly score.\n  • No labels required at inference time; the model is purely distribution-based.\n\nWHY ISOLATION FOREST FOR THIS TASK\n  • Random binary splits isolate outliers in far fewer cuts than normal points.\n  • Scales to 50k+ rows with minimal memory.\n  • Does not assume any cluster shape or Gaussian distribution — ideal for\n    SOCMINT data that mixes integer counts, ratios, and temporal signals.\n  • Catches *novel* attack compositions, not just high-severity ones — a\n    coordi

In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH   = Path("/Users/amitsingh/ML_Projects/WarfareAI/datasets/socint_alerts.csv")
OUTPUT_DIR  = Path("/Users/amitsingh/ML_Projects/WarfareAI/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH   = OUTPUT_DIR / "model4_isolation_forest.joblib"
RESULTS_PATH = OUTPUT_DIR / "model4_scored_alerts.csv"
REPORT_PATH  = OUTPUT_DIR / "/Users/amitsingh/ML_Projects/WarfareAI/datasets/SOCMINT/model4_training_report.json"

# ── Hyper-parameters ───────────────────────────────────────────────────────────
IF_N_ESTIMATORS  = 300    # more trees → more stable scores on tail events
IF_MAX_SAMPLES   = "auto" # auto = min(256, n_train_samples)
IF_CONTAMINATION = 0.05   # expected fraction of anomalies in the full dataset
                          # (used only to threshold scores into binary flag)
IF_RANDOM_STATE  = 42
N_JOBS           = -1     # parallelise across all cores


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# 1. LOAD & PARSE
# ──────────────────────────────────────────────────────────────────────────────
print("▶ Loading data …")
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
print(f"  Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"  Label distribution:\n{df['threat_label'].value_counts().to_string()}\n")



▶ Loading data …
  Loaded 50,000 rows, 18 columns
  Label distribution:
threat_label
MODERATE    14914
LOW         14038
HIGH        13515
CRITICAL     7533



In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
#    Every feature is commented with *why* it is useful to an Isolation Forest
#    on SOCMINT data — not just what it measures.
# ──────────────────────────────────────────────────────────────────────────────
print("▶ Engineering features …")

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Accepts the raw dataframe (or any subset of it).
    Returns a new dataframe with all engineered features ready for the model.
    Does NOT mutate the input.
    """
    X = df.copy()

    # ── 2A. TEMPORAL FEATURES ─────────────────────────────────────────────────
    # Isolation forests are time-agnostic; we encode temporal context as numbers
    # so they become axes in feature space.

    X["hour_of_day"] = X["timestamp"].dt.hour
    # Unusual hours (03:00 – 05:00 local) correlate with covert PSYOP launches.
    # A campaign that's normal in composition but fires at 04:00 is anomalous.

    X["day_of_week"] = X["timestamp"].dt.dayofweek
    # Weekend spikes in propaganda with no informant cover can be novel.

    X["month"] = X["timestamp"].dt.month
    # Seasonal baseline shifts (monsoon border tensions, election cycles, etc.)

    X["is_weekend"] = (X["day_of_week"] >= 5).astype(int)
    # Binary: weekend vs weekday. Simple but orthogonal to other features.

    X["hour_sin"] = np.sin(2 * np.pi * X["hour_of_day"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["hour_of_day"] / 24)
    # Cyclical encoding of hour so distance between 23:00 and 00:00 is small,
    # not 23 units apart. Prevents the forest from treating midnight as an edge.

    X["month_sin"] = np.sin(2 * np.pi * X["month"] / 12)
    X["month_cos"] = np.cos(2 * np.pi * X["month"] / 12)
    # Same cyclical treatment for month.

    # ── 2B. REGION ENCODING ───────────────────────────────────────────────────
    # Ordinal encode region so the tree can split on it numerically.
    # One-hot would create collinear columns — undesirable for IF.

    region_order = ["india-pak", "india-china", "india-arunachal", "india-bangladesh"]
    region_map   = {r: i for i, r in enumerate(region_order)}
    X["region_enc"] = X["region"].map(region_map).fillna(-1).astype(int)
    # A novel campaign in an unusual region for its alert type is anomalous.

    # ── 2C. RAW COUNT FEATURES (already in data, kept as-is) ─────────────────
    # n_protests, n_propaganda, n_informants, n_total_alerts,
    # n_high_sev, n_med_sev, n_low_sev, alerts_near_border, n_active_scenarios
    # These are the primary axes of the alert composition.

    # ── 2D. SEVERITY COMPOSITION FEATURES ────────────────────────────────────

    X["med_sev_ratio"] = (
        X["n_med_sev"] / X["n_total_alerts"].replace(0, np.nan)
    ).fillna(0)
    # Complement to high_sev_ratio. A campaign that parks everything at MEDIUM
    # avoids high-sev triggers but may still be coordinated.

    X["low_sev_ratio"] = (
        X["n_low_sev"] / X["n_total_alerts"].replace(0, np.nan)
    ).fillna(0)
    # High low_sev_ratio with high total_alerts = noise-flooding pattern.

    X["sev_gini"] = 1 - (
        X["high_sev_ratio"]**2
        + X["med_sev_ratio"]**2
        + X["low_sev_ratio"]**2
    )
    # Severity Gini impurity. Pure distributions (all HIGH or all LOW) are
    # structurally unusual vs. mixed normal traffic. Range [0, 0.667].

    X["sev_entropy"] = -(
        X["high_sev_ratio"].clip(1e-9)  * np.log(X["high_sev_ratio"].clip(1e-9))
        + X["med_sev_ratio"].clip(1e-9) * np.log(X["med_sev_ratio"].clip(1e-9))
        + X["low_sev_ratio"].clip(1e-9) * np.log(X["low_sev_ratio"].clip(1e-9))
    )
    # Shannon entropy of severity distribution. Coordinated campaigns often
    # have unusually *low* entropy (concentrated at one tier).

    # ── 2E. ALERT TYPE COMPOSITION FEATURES ───────────────────────────────────

    X["protest_ratio"] = (
        X["n_protests"] / X["n_total_alerts"].replace(0, np.nan)
    ).fillna(0)
    # Share of physical protest alerts. Novel hybrid campaigns may spike this
    # independently of propaganda or informant channels.

    X["type_entropy"] = -(
        X["protest_ratio"].clip(1e-9)   * np.log(X["protest_ratio"].clip(1e-9))
        + X["propaganda_ratio"].clip(1e-9) * np.log(X["propaganda_ratio"].clip(1e-9))
        + X["informant_ratio"].clip(1e-9)  * np.log(X["informant_ratio"].clip(1e-9))
    )
    # Shannon entropy of *alert type*. Attacks that over-index on a single
    # channel (pure propaganda, pure protest) are novel vs. balanced LOW days.

    X["protest_propaganda_diff"] = X["protest_ratio"] - X["propaganda_ratio"]
    # Signed difference: large positive → physical-dominant; large negative →
    # info-war dominant. Extreme values in either direction are anomalous.

    X["informant_dominance"] = (
        X["informant_ratio"] - (X["protest_ratio"] + X["propaganda_ratio"]) / 2
    )
    # How much the informant channel dominates over the kinetic+propaganda mean.
    # Unusual when informants fire without corresponding ground activity.

    # ── 2F. CROSS-CHANNEL INTERACTION FEATURES ────────────────────────────────
    # These capture *joint* behaviour that neither channel alone would reveal.

    X["protest_x_propaganda"] = X["n_protests"] * X["n_propaganda"]
    # Product of protest and propaganda counts. Coordinated hybrid operations
    # that simultaneously flood both channels are anomalous relative to LOW days
    # where one or the other is elevated but rarely both together.

    X["sev_x_propaganda"] = X["mean_severity_score"] * X["propaganda_ratio"]
    # High-severity alerts that are also propaganda-heavy signal a specific
    # campaign pattern not usually seen in normal LOW traffic.

    X["border_x_high_sev"] = X["alerts_near_border"] * X["n_high_sev"]
    # Border-proximal HIGH alerts. During LOW days this product is near zero.
    # A spike with a novel ratio is a strong isolation signal.

    X["scenario_x_total"] = X["n_active_scenarios"] * X["n_total_alerts"]
    # Multiple active scenarios AND high total volume = potential multi-front
    # coordination — rare in pure LOW baseline.

    X["socint_x_sev"] = X["socint_score"] * X["mean_severity_score"]
    # Product of composite SOCINT score with mean severity. Captures cases
    # where the intelligence signal is amplified by the severity distribution.

    # ── 2G. VOLUME ANOMALY FEATURES ───────────────────────────────────────────

    X["total_alert_sq"] = X["n_total_alerts"] ** 2
    # Squared volume. Isolation forests split linearly; squaring makes extreme
    # volumes (volume surges) easier to isolate in fewer cuts.

    X["sev_weighted_total"] = (
        X["n_high_sev"] * 3 + X["n_med_sev"] * 2 + X["n_low_sev"] * 1
    )
    # Severity-weighted alert count. Comparable to an internal risk score
    # but computed without the modelled socint_score — independent signal.

    X["sev_ratio_spread"] = X["high_sev_ratio"] - X["low_sev_ratio"]
    # Spread between extremes of the severity distribution.
    # In LOW days both extremes are low; a spread that is unusually positive
    # (HIGH floods) or negative (LOW floods) is anomalous.

    X["alerts_per_scenario"] = (
        X["n_total_alerts"] / (X["n_active_scenarios"] + 1)
    )
    # Average alerts per active scenario. Unusually high = single-scenario
    # overload; unusually low = many quiet scenarios all ticking simultaneously.

    X["border_density"] = (
        X["alerts_near_border"] / (X["n_total_alerts"].replace(0, np.nan))
    ).fillna(0)
    # Fraction of all alerts that occur near the border. Pure-border campaigns
    # with high density are novel vs. typical LOW traffic.

    X["zero_alert_flag"] = (X["n_total_alerts"] == 0).astype(int)
    # Binary flag for complete silence. Sudden full-silence periods can
    # themselves be a campaign tactic (communication blackout before surge).

    X["channel_concentration"] = (
        pd.concat([X["protest_ratio"], X["propaganda_ratio"], X["informant_ratio"]], axis=1)
        .max(axis=1)
    )
    # Maximum share held by any single channel. High = mono-channel campaign;
    # novel because LOW days tend to show balanced multi-channel activity.

    return X


# Apply feature engineering to the full dataset
df_feat = engineer_features(df)

# Define the exact feature columns the model will consume
FEATURE_COLS = [
    # ── temporal ──────────────────────────────────────────────────────────────
    "hour_sin", "hour_cos",            # cyclical hour encoding
    "month_sin", "month_cos",          # cyclical month encoding
    "day_of_week",                     # Mon=0 … Sun=6
    "is_weekend",                      # binary weekday vs. weekend

    # ── spatial ───────────────────────────────────────────────────────────────
    "region_enc",                      # ordinal region identifier
    "alerts_near_border",              # raw border-proximity count
    "border_density",                  # fraction of alerts at border

    # ── raw counts ────────────────────────────────────────────────────────────
    "n_protests",
    "n_propaganda",
    "n_informants",
    "n_total_alerts",
    "n_high_sev",
    "n_med_sev",
    "n_low_sev",
    "n_active_scenarios",

    # ── severity composition ───────────────────────────────────────────────────
    "high_sev_ratio",                  # fraction of alerts that are HIGH
    "med_sev_ratio",                   # fraction that are MED
    "low_sev_ratio",                   # fraction that are LOW
    "mean_severity_score",             # pre-computed mean severity
    "sev_gini",                        # Gini impurity of severity distribution
    "sev_entropy",                     # Shannon entropy of severity distribution
    "sev_ratio_spread",                # high_ratio - low_ratio
    "sev_weighted_total",              # 3·HIGH + 2·MED + 1·LOW
    "total_alert_sq",                  # volume² for non-linear isolation

    # ── alert type composition ─────────────────────────────────────────────────
    "protest_ratio",                   # fraction of alerts that are protests
    "propaganda_ratio",                # fraction that are propaganda
    "informant_ratio",                 # fraction from informants
    "type_entropy",                    # Shannon entropy over alert type
    "protest_propaganda_diff",         # signed channel dominance
    "informant_dominance",             # informant vs mean(protest, propaganda)
    "channel_concentration",           # max single-channel share

    # ── cross-channel interactions ─────────────────────────────────────────────
    "protest_x_propaganda",            # joint escalation product
    "sev_x_propaganda",                # severity × propaganda ratio
    "border_x_high_sev",               # border proximity × HIGH count
    "scenario_x_total",                # scenario count × total volume
    "socint_x_sev",                    # SOCINT score × mean severity

    # ── volume / load ──────────────────────────────────────────────────────────
    "alerts_per_scenario",             # volume normalised by active scenarios
    "border_density",                  # re-included as explicit volume ratio
    "zero_alert_flag",                 # complete silence indicator

    # ── composite score ────────────────────────────────────────────────────────
    "socint_score",                    # existing model-derived composite signal
]

# Deduplicate (border_density appears twice above — intentional for clarity)
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))

print(f"  Total engineered features: {len(FEATURE_COLS)}")


▶ Engineering features …
  Total engineered features: 41


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# 3. TRAIN / INFERENCE SPLIT
#    Training: LOW-threat rows only  →  learn what "normal" looks like
#    Inference: the entire dataset   →  flag anything that doesn't fit
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Splitting train (LOW) vs inference (full) …")

train_mask = df["threat_label"] == "LOW"
df_train   = df_feat[train_mask].copy()
df_all     = df_feat.copy()          # used at inference time

X_train = df_train[FEATURE_COLS].values
X_all   = df_all[FEATURE_COLS].values

print(f"  Training rows (LOW only) : {len(X_train):,}")
print(f"  Inference rows (all)     : {len(X_all):,}")

# Sanity check for NaN / Inf — IF does not handle them gracefully
assert not np.isnan(X_train).any(), "NaN found in training features"
assert not np.isinf(X_train).any(), "Inf found in training features"




▶ Splitting train (LOW) vs inference (full) …
  Training rows (LOW only) : 14,038
  Inference rows (all)     : 50,000


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# 4. BUILD & FIT THE PIPELINE
#    StandardScaler is technically optional for IF (tree-based), but it
#    makes the raw anomaly scores comparable across feature axes and ensures
#    that high-magnitude features (n_total_sq can be >600) don't dominate the
#    random split distribution, which would bias feature selection in the trees.
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Building sklearn Pipeline: StandardScaler → IsolationForest …")

pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
        # Fit on LOW-only data so the scaler's μ/σ reflect normal baseline.
        # This means HIGH/CRITICAL alerts will be many σ away on magnitude
        # features, which is intentional — we want them to appear far from
        # the learned normal distribution.
    ),
    (
        "iforest",
        IsolationForest(
            n_estimators=IF_N_ESTIMATORS,
            # Number of trees. 300 gives stable scores even on tail events;
            # diminishing returns beyond ~500.

            max_samples=IF_MAX_SAMPLES,
            # Subsample size per tree. "auto" → min(256, n_samples).
            # Smaller subsamples = more isolation variance per tree, which
            # is desirable — each tree sees a different view of the data.

            contamination=IF_CONTAMINATION,
            # Used ONLY to compute the decision threshold (offset_).
            # Does not affect how trees are built. Set to 0.05 = we expect
            # ~5% of all alerts to be anomalous; adjust if domain knowledge
            # suggests otherwise.

            max_features=1.0,
            # Fraction of features to consider at each split. 1.0 = all
            # features; can be reduced (e.g. 0.8) to add diversity.

            bootstrap=False,
            # No bootstrap — each tree gets a fresh random subsample drawn
            # WITHOUT replacement, which is the original Isolation Forest spec.

            n_jobs=N_JOBS,
            random_state=IF_RANDOM_STATE,
        )
    )
])

print("  Fitting on LOW-threat training data …")
pipeline.fit(X_train)
print("  ✓ Pipeline fitted.")



▶ Building sklearn Pipeline: StandardScaler → IsolationForest …
  Fitting on LOW-threat training data …
  ✓ Pipeline fitted.


In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# 5. INFERENCE ON FULL DATASET
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Running inference on full dataset …")

# score_samples returns the raw anomaly score (negative of mean path length).
# Lower (more negative) = more anomalous. We negate so higher = more anomalous.
raw_scores = -pipeline.score_samples(X_all)

# Binary flag: 1 = anomalous, using the contamination threshold baked into
# the fitted model (pipeline[-1].offset_).
binary_preds = (pipeline.predict(X_all) == -1).astype(int)
# IsolationForest.predict() returns -1 for outliers and +1 for inliers.

# Percentile rank 0–100 for dashboard display
percentile_scores = pd.Series(raw_scores).rank(pct=True).values * 100

# Stitch results back onto the original dataframe
df_results = df.copy()
df_results["anomaly_score"]    = raw_scores
df_results["if_flag"]          = binary_preds
df_results["percentile_score"] = percentile_scores.round(2)

print(f"  Flagged anomalies  : {binary_preds.sum():,} "
      f"({binary_preds.mean()*100:.1f}% of all alerts)")
print(f"  Score range        : [{raw_scores.min():.4f}, {raw_scores.max():.4f}]")



▶ Running inference on full dataset …
  Flagged anomalies  : 35,194 (70.4% of all alerts)
  Score range        : [0.4147, 0.7344]


In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# 6. EVALUATION (proxy metrics, since IF is unsupervised)
#    We use the ground-truth threat labels as PROXY only — to check that
#    the anomaly scores broadly correlate with known high-threat conditions.
#    These metrics are diagnostic, NOT optimisation targets.
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Computing proxy evaluation metrics …")

# Binary proxy: treat CRITICAL as "ground-truth anomaly"
y_critical = (df["threat_label"] == "CRITICAL").astype(int)
auc_roc_crit  = roc_auc_score(y_critical, raw_scores)
auc_pr_crit   = average_precision_score(y_critical, raw_scores)

# Binary proxy: treat HIGH + CRITICAL as anomaly
y_high = (df["threat_label_int"] >= 2).astype(int)
auc_roc_high  = roc_auc_score(y_high, raw_scores)
auc_pr_high   = average_precision_score(y_high, raw_scores)

# Score distribution by label (sanity check — should increase with threat level)
score_by_label = df_results.groupby("threat_label")["anomaly_score"].agg(
    ["mean", "median", "std"]
).round(4)

print(f"\n  Proxy ROC-AUC (CRITICAL vs rest)     : {auc_roc_crit:.4f}")
print(f"  Proxy AP      (CRITICAL vs rest)     : {auc_pr_crit:.4f}")
print(f"  Proxy ROC-AUC (HIGH+CRITICAL vs rest): {auc_roc_high:.4f}")
print(f"  Proxy AP      (HIGH+CRITICAL vs rest): {auc_pr_high:.4f}")
print(f"\n  Mean anomaly score by threat label:\n{score_by_label.to_string()}")



▶ Computing proxy evaluation metrics …

  Proxy ROC-AUC (CRITICAL vs rest)     : 0.9552
  Proxy AP      (CRITICAL vs rest)     : 0.7772
  Proxy ROC-AUC (HIGH+CRITICAL vs rest): 0.9730
  Proxy AP      (HIGH+CRITICAL vs rest): 0.9620

  Mean anomaly score by threat label:
                mean  median     std
threat_label                        
CRITICAL      0.7081  0.7086  0.0092
HIGH          0.6883  0.6918  0.0179
LOW           0.4841  0.4770  0.0368
MODERATE      0.6245  0.6340  0.0490


In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# 7. FEATURE IMPORTANCE (permutation-based, approximated on a validation subset)
#    Standard IF does not expose built-in feature importances.
#    We approximate via permutation importance on the training set using the
#    raw anomaly score as the pseudo-target.
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Computing permutation feature importances (this may take ~30s) …")

from sklearn.base import BaseEstimator, RegressorMixin

class IFScoreRegressor(BaseEstimator, RegressorMixin):
    """Thin sklearn-compatible wrapper so permutation_importance can call .score()."""
    def __init__(self, fitted_pipeline):
        self.fitted_pipeline = fitted_pipeline
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return -self.fitted_pipeline.score_samples(X)
    def score(self, X, y):
        # Use correlation between predicted anomaly scores and raw_scores
        # as a proxy for how much permuting a feature degrades ordering.
        preds = self.predict(X)
        return float(np.corrcoef(preds, y)[0, 1])

# Use a stratified sample for speed — 5000 rows from LOW training set
rng       = np.random.default_rng(42)
idx_sub   = rng.choice(len(X_train), size=min(5000, len(X_train)), replace=False)
X_sub     = X_train[idx_sub]
y_sub     = raw_scores[train_mask.values][idx_sub]   # aligned scores

wrapped   = IFScoreRegressor(pipeline)
perm_res  = permutation_importance(
    wrapped, X_sub, y_sub,
    n_repeats=10,
    n_jobs=N_JOBS,
    random_state=42
)

feat_imp_df = pd.DataFrame({
    "feature"         : FEATURE_COLS,
    "importance_mean" : perm_res.importances_mean,
    "importance_std"  : perm_res.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print(f"\n  Top 15 most important features:")
print(feat_imp_df.head(15).to_string(index=False))



▶ Computing permutation feature importances (this may take ~30s) …

  Top 15 most important features:
             feature  importance_mean  importance_std
  alerts_near_border         0.021328        0.000274
          is_weekend         0.020686        0.000390
     zero_alert_flag         0.018595        0.000508
protest_x_propaganda         0.018108        0.000393
      border_density         0.017115        0.000272
            sev_gini         0.014515        0.000231
       low_sev_ratio         0.012816        0.000146
      high_sev_ratio         0.012376        0.000249
         sev_entropy         0.012087        0.000156
       med_sev_ratio         0.012068        0.000133
   border_x_high_sev         0.011589        0.000165
           n_med_sev         0.011416        0.000228
       protest_ratio         0.011242        0.000131
    sev_x_propaganda         0.010493        0.000192
          n_high_sev         0.010229        0.000146


In [11]:
# ──────────────────────────────────────────────────────────────────────────────
# 8. VISUALISATIONS
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Generating plots …")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Model 4 — Isolation Forest SOCMINT Anomaly Detector", fontsize=15, fontweight="bold")

# ── 8a. Score distributions by threat label ───────────────────────────────────
ax = axes[0, 0]
label_order = ["LOW", "MODERATE", "HIGH", "CRITICAL"]
colors      = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
for lbl, col in zip(label_order, colors):
    subset = df_results[df_results["threat_label"] == lbl]["anomaly_score"]
    ax.hist(subset, bins=60, alpha=0.55, label=lbl, color=col, density=True)
ax.set_xlabel("Anomaly Score (higher = more anomalous)")
ax.set_ylabel("Density")
ax.set_title("Anomaly Score Distribution by Threat Label")
ax.legend()
ax.grid(alpha=0.3)

# ── 8b. Box plot of scores by label ──────────────────────────────────────────
ax = axes[0, 1]
data_by_label = [df_results[df_results["threat_label"] == l]["anomaly_score"].values
                 for l in label_order]
bp = ax.boxplot(data_by_label, labels=label_order, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, col in zip(bp["boxes"], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.set_ylabel("Anomaly Score")
ax.set_title("Score Spread by Threat Label")
ax.grid(axis="y", alpha=0.3)

# ── 8c. Feature importances (top 20) ─────────────────────────────────────────
ax = axes[1, 0]
top20 = feat_imp_df.head(20)
bars = ax.barh(top20["feature"][::-1], top20["importance_mean"][::-1],
               xerr=top20["importance_std"][::-1],
               color="#3498db", alpha=0.8, ecolor="#2980b9", capsize=3)
ax.set_xlabel("Permutation Importance (correlation drop)")
ax.set_title("Top 20 Feature Importances")
ax.grid(axis="x", alpha=0.3)

# ── 8d. Flagged anomalies per region ─────────────────────────────────────────
ax = axes[1, 1]
flag_by_region = df_results.groupby("region")["if_flag"].agg(["sum", "count"])
flag_by_region["flag_rate"] = flag_by_region["sum"] / flag_by_region["count"]
flag_by_region = flag_by_region.sort_values("flag_rate", ascending=True)
ax.barh(flag_by_region.index, flag_by_region["flag_rate"] * 100,
        color="#e67e22", alpha=0.8)
ax.set_xlabel("Anomaly Flag Rate (%)")
ax.set_title("Anomaly Flag Rate by Region")
ax.grid(axis="x", alpha=0.3)
for i, (_, row) in enumerate(flag_by_region.iterrows()):
    ax.text(row["flag_rate"] * 100 + 0.1, i,
            f"{int(row['sum'])} / {int(row['count'])}", va="center", fontsize=9)

plt.tight_layout()
plot_path = OUTPUT_DIR / "model4_diagnostics.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  ✓ Saved diagnostics plot → {plot_path}")



▶ Generating plots …
  ✓ Saved diagnostics plot → /Users/amitsingh/ML_Projects/WarfareAI/outputs/model4_diagnostics.png


In [12]:
# ──────────────────────────────────────────────────────────────────────────────
# 9. SAVE ARTEFACTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n▶ Saving artefacts …")

# 9a. Serialised pipeline (scaler + IF model)
joblib.dump(
    {"pipeline": pipeline, "feature_cols": FEATURE_COLS},
    MODEL_PATH
)
print(f"  ✓ Model pipeline → {MODEL_PATH}")

# 9b. Full scored CSV
df_results.to_csv(RESULTS_PATH, index=False)
print(f"  ✓ Scored alerts  → {RESULTS_PATH}")

# 9c. Human-readable training report
report = {
    "model"            : "IsolationForest",
    "train_label"      : "LOW",
    "n_train"          : int(X_train.shape[0]),
    "n_features"       : len(FEATURE_COLS),
    "feature_cols"     : FEATURE_COLS,
    "hyperparameters"  : {
        "n_estimators"  : IF_N_ESTIMATORS,
        "max_samples"   : IF_MAX_SAMPLES,
        "contamination" : IF_CONTAMINATION,
        "random_state"  : IF_RANDOM_STATE,
    },
    "proxy_metrics"    : {
        "roc_auc_critical"          : round(auc_roc_crit,  4),
        "avg_precision_critical"    : round(auc_pr_crit,   4),
        "roc_auc_high_plus"         : round(auc_roc_high,  4),
        "avg_precision_high_plus"   : round(auc_pr_high,   4),
    },
    "flagged_total"    : int(binary_preds.sum()),
    "flagged_pct"      : round(float(binary_preds.mean() * 100), 2),
    "score_stats"      : {
        "min"  : round(float(raw_scores.min()),  4),
        "max"  : round(float(raw_scores.max()),  4),
        "mean" : round(float(raw_scores.mean()), 4),
        "std"  : round(float(raw_scores.std()),  4),
    },
    "score_by_label"   : score_by_label.to_dict(),
    "top10_features"   : feat_imp_df.head(10)["feature"].tolist(),
}

with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print(f"  ✓ Training report → {REPORT_PATH}")



▶ Saving artefacts …
  ✓ Model pipeline → /Users/amitsingh/ML_Projects/WarfareAI/outputs/model4_isolation_forest.joblib
  ✓ Scored alerts  → /Users/amitsingh/ML_Projects/WarfareAI/outputs/model4_scored_alerts.csv
  ✓ Training report → /Users/amitsingh/ML_Projects/WarfareAI/datasets/SOCMINT/model4_training_report.json


In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# 10. INFERENCE HELPER  (copy-paste into dashboard / API server)
# ──────────────────────────────────────────────────────────────────────────────
INFERENCE_SNIPPET = '''
# ── Model 4 Inference ─────────────────────────────────────────────────────────
import joblib, pandas as pd, numpy as np
from pathlib import Path

artefact    = joblib.load("model4_isolation_forest.joblib")
pipeline    = artefact["pipeline"]
FEATURE_COLS = artefact["feature_cols"]

def score_alerts(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    df_raw : any slice of socint_alerts (or new live alerts with same schema).
    Returns the input dataframe with three new columns:
        anomaly_score    – higher = more anomalous vs LOW baseline
        if_flag          – 1 = flagged as novel/anomalous
        percentile_score – 0-100 rank within this batch
    """
    df = engineer_features(df_raw)        # same function defined in training script
    X  = df[FEATURE_COLS].values

    scores        = -pipeline.score_samples(X)
    flags         = (pipeline.predict(X) == -1).astype(int)
    pctile        = pd.Series(scores).rank(pct=True).values * 100

    df_raw = df_raw.copy()
    df_raw["anomaly_score"]    = scores
    df_raw["if_flag"]          = flags
    df_raw["percentile_score"] = pctile.round(2)
    return df_raw
'''

print("\n" + "═" * 70)
print("INFERENCE SNIPPET (copy into dashboard / Streamlit page):")
print("═" * 70)
print(INFERENCE_SNIPPET)

print("\n✅  Model 4 training complete.")
print(f"   Pipeline   : {MODEL_PATH}")
print(f"   Scored CSV : {RESULTS_PATH}")
print(f"   Report     : {REPORT_PATH}")
print(f"   Plots      : {plot_path}")


══════════════════════════════════════════════════════════════════════
INFERENCE SNIPPET (copy into dashboard / Streamlit page):
══════════════════════════════════════════════════════════════════════

# ── Model 4 Inference ─────────────────────────────────────────────────────────
import joblib, pandas as pd, numpy as np
from pathlib import Path

artefact    = joblib.load("model4_isolation_forest.joblib")
pipeline    = artefact["pipeline"]
FEATURE_COLS = artefact["feature_cols"]

def score_alerts(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    df_raw : any slice of socint_alerts (or new live alerts with same schema).
    Returns the input dataframe with three new columns:
        anomaly_score    – higher = more anomalous vs LOW baseline
        if_flag          – 1 = flagged as novel/anomalous
        percentile_score – 0-100 rank within this batch
    """
    df = engineer_features(df_raw)        # same function defined in training script
    X  = df[FEATURE_COLS].values

    sco